# Module 4 — Forward Kinematics using Denavit–Hartenberg Parameters
## Unit 8 — Mini Project: From Joints to the Fruit
### Lesson 8.2 — Building the Arm's DH Model

*Physical AI Curriculum · capstone notebook. Run **Kernel → Restart & Run All**.*

In [ ]:
import numpy as np
from functools import reduce
def dh(theta,d,a,al):
    ct,st,ca,sa=np.cos(theta),np.sin(theta),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1.0]])
def dh_fk(table,cfg):
    f=[dh(q,d,a,al) for (d,a,al,_),q in zip(table,cfg)]
    return reduce(lambda A,B:A@B,f,np.eye(4))
def se3(R,t):
    T=np.eye(4);T[:3,:3]=R;T[:3,3]=t;return T
def apply(M,P): return (M@np.r_[np.asarray(P,float),1.0])[:3]
def capstone(cfg,P_cam,T_base_cam,table):
    T0n=dh_fk(table,cfg)
    P_base=apply(T_base_cam,P_cam)
    P_grip=apply(np.linalg.inv(T0n),P_base)
    move=P_base-T0n[:3,3]
    return {'T0n':T0n,'P_base':P_base,'P_grip':P_grip,'move':move}
arm3=[(0.1,0,np.radians(90),'r'),(0,0.4,0,'r'),(0,0.3,0,'r')]
T_base_cam=se3(np.eye(3),[0.1,0,0.5])
out=capstone([0,np.radians(30),np.radians(60)],[0.05,0,0.40],T_base_cam,arm3)
print('gripper pos',np.round(out['T0n'][:3,3],3))
print('fruit base ',np.round(out['P_base'],3),' move',np.round(out['move'],3))

## Coding Exercise (§8)
Run the assembled pipeline and print all four outputs; add a reachability gate.

In [ ]:
# YOUR CODE HERE
import numpy as np
assert np.allclose(out['P_base'],[0.15,0,0.90])
def reachable(P_base,reach_max=0.7,riser=0.1):
    d=np.hypot(np.hypot(P_base[0],P_base[1]),P_base[2]-riser)
    return d<=reach_max+1e-9
print('fruit rel gripper',np.round(out['P_grip'],3))
print('reachable?', reachable(out['P_base']))
print('capstone pipeline runs end to end.')

## Check your work

In [ ]:
import numpy as np
assert out['T0n'].shape==(4,4) and out['move'].shape==(3,)
print('All checks passed.')